### Imports

In [4]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv

#Configurar caminho para importar funções do src/utils.py
notebook_dir = Path.cwd().resolve()
project_root = notebook_dir.parents[1] if notebook_dir.name == 'notebooks' else notebook_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import iniciar_cessao_aws, carregar_parquet_s3, listar_arquivos_s3


### Carregar Dados

In [5]:

# 2. Conectar ao S3 com credenciais do .env
load_dotenv(find_dotenv())
BUCKET_NAME = os.getenv('BUCKET_NAME')

session = iniciar_cessao_aws()
s3_client = session.client('s3')

print(f"Conectado ao S3 no bucket: {BUCKET_NAME}")

# 3. Carregar o Parquet da camada Silver
# Recomendamos carregar 'reviews_enriquecidos' (já contém as mensagens + metadados de pedidos e produtos)
df_rag = carregar_parquet_s3(
    s3_client=s3_client,
    bucket=BUCKET_NAME,
    nome_tabela='reviews_enriquecidos',
    camada='silver'
)

# Caso queira carregar apenas o 'mensagens':
# df_mensagens = carregar_parquet_s3(s3_client=s3_client, bucket=BUCKET_NAME, nome_tabela='mensagens', camada='silver')

print(f"\n Base Silver carregada com sucesso! Dimensões: {df_rag.shape}")
display(df_rag.head(3))


Conectado ao S3 no bucket: postech-challenge-datascience-004
Lendo: s3://postech-challenge-datascience-004/silver/reviews_enriquecidos.parquet

 Base Silver carregada com sucesso! Dimensões: (37257, 21)


,order_id,review,avaliacao,tratamento_1,tratamento_2,tratamento_3,tratamento_4,review_id,review_score,order_status,...,dias_atraso,entregue_com_atraso,product_category_name,price,freight_value,qtd_itens,review_comment_title,texto_completo,review_rag,qtd_palavras
0,658677c97b385a9be170737859d3511b,Recebi bem antes do prazo estipulado.,positiva,Recebi bem antes prazo estipulado.,Recebi bem antes prazo estipulado,Recebi bem antes prazo estipulado,recebi bem antes prazo estipulado,e64fb393e7b32834bb789ff8bb30750e,5,delivered,...,-20.0,False,ferramentas_jardim,179.99,42.85,1.0,NaN,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado.,6
1,8e6bfb81e283fa7e4f11123a3fb894f1,Parabéns lojas lannister adorei comprar pela I...,positiva,Parabéns lojas lannister adorei comprar Intern...,Parabéns lojas lannister adorei comprar Intern...,Parabens lojas lannister adorei comprar Intern...,parabens lojas lannister adorei comprar intern...,f7c4243c7fe1938f181bec41a392bdeb,5,delivered,...,-9.0,False,esporte_lazer,1199.00,134.25,1.0,NaN,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...,15
2,b9bf720beb4ab3728760088589c62129,aparelho eficiente. no site a marca do aparelh...,positiva,aparelho eficiente. site marca aparelho impres...,aparelho eficiente site marca aparelho impress...,aparelho eficiente site marca aparelho impress...,aparelho eficiente site marca aparelho impress...,8670d52e15e00043ae7de4c01cc2fe06,4,delivered,...,-16.0,False,eletroportateis,586.56,26.69,1.0,recomendo,recomendo. aparelho eficiente. no site a marca...,recomendo. aparelho eficiente. no site a marca...,31


Apesar de ter feito diversos tratamentos de dados, para os modelos atuais não precisamos nos preocupar com a limpeza de pontuação 

In [14]:
df = df_rag[['review_id','review_score','order_id','product_category_name','entregue_com_atraso','dias_atraso','price','avaliacao','review_rag']]
df.isna().sum()

review_id                   0
review_score                0
order_id                    0
product_category_name     501
entregue_com_atraso         0
dias_atraso              1830
price                     501
avaliacao                   0
review_rag                  0
dtype: int64

Criação de um document para transformarmos o dataframe em uma estrutura que o LangChain Consiga utilizar, para isso primeiro tratamos os nulos. <br>
fazemos uma amostra controlada para que possamos gerar os embeddings sem consumir muito da nossa chave e criaremos o document com o page_content e o metadata

In [20]:
from langchain_core.documents import Document

# Tratar valores nulos nos metadados para evitar erros no ChromaDB
df_clean = df.copy()
df_clean['product_category_name'] = df_clean['product_category_name'].fillna('outros')
df_clean['dias_atraso'] = df_clean['dias_atraso'].fillna(0.0)
df_clean['entregue_com_atraso'] = df_clean['entregue_com_atraso'].fillna(False).astype(bool)
df_clean['price'] = df_clean['price'].fillna(0.0)

# para testar o fluxo com rapidez e sem gastar cota, antes de vetorizar os 37k vamos usar 300 para teste no freetier do gemini
df_sample = df_clean.sample(n=min(300, len(df_clean)), random_state=42)

# Converter para objetos Document do LangChain
documentos = []
for _, row in df_sample.iterrows():
    # Usamos o texto natural completo (review_rag ou texto_completo)
    conteudo = str(row['review_rag']) if pd.notna(row['review_rag']) else str(row['review'])
    
    # Metadados valiosos para filtros e citações de evidências
    metadados = {
        "review_id": str(row['review_id']),
        "order_id": str(row['order_id']),
        "review_score": int(row['review_score']),
        "product_category_name": str(row['product_category_name']),
        "entregue_com_atraso": bool(row['entregue_com_atraso']),
        "dias_atraso": float(row['dias_atraso']),
        "avaliacao": str(row['avaliacao'])
    }
    documentos.append(Document(page_content=conteudo, metadata=metadados))

print(f"Total de documentos criados: {len(documentos)}")
print("\nExemplo do primeiro documento:")
print("Conteúdo:", documentos[0].page_content)
print("Metadados:", documentos[0].metadata)



Total de documentos criados: 300

Exemplo do primeiro documento:
Conteúdo: Recomendo. Entrega no prazo e linda cortina!
Metadados: {'review_id': '7e647d150683df9d45d55c702b4c1a7a', 'order_id': '5ce8ed74908cd24bd6b5138292ec7b04', 'review_score': 5, 'product_category_name': 'cama_mesa_banho', 'entregue_com_atraso': False, 'dias_atraso': -7.0, 'avaliacao': 'positiva'}


Faremos agora os Embeddings a partir do nosso documento 
O  page_content é enviado e retornara uma lista com a tradução semantica do texto em um vetor 
<br><br>
o Chroma salva os vetores e os metadados em persist_directory 
<br><br>
Com a persistencia nao é necessario reprocessar a base vetorial a cada execução  
<br><br>
Para manter no free Tier fazemos um processamento de 25 linhas e uma espera de 2 segundos 
criamos uma resiliencia para o código nao quebrar caso atinja o limite da chave RESOURCE_EXHAUSTED 

In [23]:
import time
from tqdm import tqdm
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# 2. Inicializar o Vector Store
diretorio_chroma = project_root / "fase_4" / "models" / "chroma_db"

vectorstore = Chroma(
    collection_name="olist_customer_reviews",
    embedding_function=embeddings,
    persist_directory=str(diretorio_chroma)
)

# 3. Ingestão em lotes com tratamento automático de 429
BATCH_SIZE = 25
total_lotes = len(documentos)

pbar = tqdm(range(0, total_lotes, BATCH_SIZE), desc="Indexando no ChromaDB")
for i in pbar:
    lote = documentos[i : i + BATCH_SIZE]
    sucesso = False
    tentativas = 0
    
    while not sucesso and tentativas < 5:
        try:
            vectorstore.add_documents(lote)
            sucesso = True
            time.sleep(2)  # Pausa segura entre lotes
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                pbar.set_postfix_str("Aguardando cota da API (20s)...")
                time.sleep(20)  # Aguarda o cooldown e tenta o mesmo lote novamente
                tentativas += 1
            else:
                raise e

print(f"\n Sucesso absoluto! {len(documentos)} avaliações indexadas no ChromaDB!")


Indexando no ChromaDB:   0%|          | 0/12 [00:00<?, ?it/s]

Indexando no ChromaDB: 100%|██████████| 12/12 [02:22<00:00, 11.90s/it, Aguardando cota da API (20s)...]


🎉 Sucesso absoluto! 300 avaliações indexadas no ChromaDB!


Com a base processada iremos agora iniciar o processo de recuperação dessas informações com a busca semantica 
para isso vamos configurar um Retriever 


In [24]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={"k": 5}
)


In [27]:
pergunta_teste = "Tive problemas com a entrega demorada"
docs_recuperados = retriever.invoke(pergunta_teste)
docs_recuperados

[Document(id='ba068c87-e430-4da5-8915-d51524e9c2e2', metadata={'entregue_com_atraso': False, 'dias_atraso': 0.0, 'order_id': '639136e1c9d70479de8dfa922ad2cf85', 'avaliacao': 'positiva', 'product_category_name': 'eletronicos', 'review_score': 5, 'review_id': '65e3d04aef73660b0b2145ef3d5762cd'}, page_content='A entrega atrasou'),
 Document(id='3353cb07-98cd-47ca-8103-9cc8e5c88823', metadata={'entregue_com_atraso': True, 'product_category_name': 'pet_shop', 'review_id': '95e7f613e3dfe80a79306a8c80084739', 'dias_atraso': 23.0, 'review_score': 1, 'avaliacao': 'negativa', 'order_id': '2d58df29239f4ee3887103ef2576e857'}, page_content='Muita demora na entrega'),
 Document(id='efea632c-7967-4f30-ae54-2c391e3ca118', metadata={'review_score': 1, 'dias_atraso': 3.0, 'product_category_name': 'ferramentas_jardim', 'entregue_com_atraso': True, 'avaliacao': 'negativa', 'order_id': '0bd15125e46474dc9318288ae0ea528e', 'review_id': 'b7fe825a931a313b8aa2d18e8004b328'}, page_content='O prazo para a entrega

faremos o teste para ver se a busca semântica está funcionando

In [28]:
print(f"Total de evidências recuperadas: {len(docs_recuperados)}\n")
for idx, doc in enumerate(docs_recuperados, 1):
    print(f"--- Evidência #{idx} ---")
    print(f"Review ID: {doc.metadata.get('review_id')}")
    print(f"Nota: {doc.metadata.get('review_score')} estrelas | Atraso: {doc.metadata.get('dias_atraso')} dias | Categoria: {doc.metadata.get('product_category_name')}")
    print(f"Comentário: {doc.page_content}\n")

Total de evidências recuperadas: 5

--- Evidência #1 ---
Review ID: 65e3d04aef73660b0b2145ef3d5762cd
Nota: 5 estrelas | Atraso: 0.0 dias | Categoria: eletronicos
Comentário: A entrega atrasou

--- Evidência #2 ---
Review ID: 95e7f613e3dfe80a79306a8c80084739
Nota: 1 estrelas | Atraso: 23.0 dias | Categoria: pet_shop
Comentário: Muita demora na entrega

--- Evidência #3 ---
Review ID: b7fe825a931a313b8aa2d18e8004b328
Nota: 1 estrelas | Atraso: 3.0 dias | Categoria: ferramentas_jardim
Comentário: O prazo para a entrega já era absurdo e ainda assim conseguiram nao entregar no prazo, ainda nao recebi o produto.

--- Evidência #4 ---
Review ID: 140159c8ce8e90db15289d6637ee4b4f
Nota: 3 estrelas | Atraso: 2.0 dias | Categoria: relogios_presentes
Comentário: Ainda não recebi o produto e ontem se encerrou o prazo! Acho que quando se paga quase 30,00 de frete, o mínimo que deveriam fazer era entregar a tempo.

--- Evidência #5 ---
Review ID: 0e2db9f3094b64e23c8cef7a3b6fb88b
Nota: 1 estrelas | Atr

## implementação do RAG

In [29]:
from langchain_core.prompts import ChatPromptTemplate

template_rag = """Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações do e-commerce Olist.
Sua missão é responder à pergunta do gestor baseando-se ESTRITAMENTE nas avaliações de clientes fornecidas no contexto abaixo.

Diretrizes obrigatórias:
1. Responda fundamentando-se APENAS nas evidências fornecidas no contexto. Não invente ou presuma fatos que não estejam nos comentários.
2. Sempre cite os IDs das avaliações [Review ID: ...] que sustentam cada constatação ou insight apresentado.
3. Se o contexto não contiver informações suficientes para responder ou se a pergunta for fora de escopo, responda educadamente: "Não foram encontradas avaliações suficientes na base de conhecimento sobre este assunto."

Contexto com as avaliações dos clientes:
{context}

Pergunta do Gestor:
{question}

Resposta Executiva Fundamentada (com evidências):"""

prompt = ChatPromptTemplate.from_template(template_rag)


In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Instanciar o modelo de linguagem Gemini Flash
llm = ChatGoogleGenerativeAI(
     model="gemini-3.5-flash-lite", #model="gemini-3.8-flash",
    temperature=0.1,  # Temperatura baixa para máxima fidelidade aos dados
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Função auxiliar para formatar os documentos no contexto incluindo os IDs
def formatar_documentos(docs):
    textos_formatados = []
    for doc in docs:
        id_review = doc.metadata.get('review_id', 'N/A')
        nota = doc.metadata.get('review_score', 'N/A')
        categoria = doc.metadata.get('product_category_name', 'N/A')
        textos_formatados.append(
            f"[Review ID: {id_review} | Nota: {nota} estrelas | Categoria: {categoria}]\n{doc.page_content}"
        )
    return "\n\n".join(textos_formatados)

#  Construção da Chain usando LCEL (padrão oficial moderno do LangChain)
rag_chain = (
    {"context": retriever | formatar_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Pipeline RAG configurado com sucesso!")


Pipeline RAG configurado com sucesso!


## fazendo as perguntas sugeridas pelo teach challenge 

In [ ]:
pergunta = "Quais são os principais problemas relatados pelos clientes sobre entrega?"
resposta = rag_chain.invoke(pergunta)
print(resposta)

Com base estritamente nas avaliações analisadas, os principais problemas relatados pelos clientes em relação à entrega dividem-se em três pontos críticos:

1. **Atraso e demora na entrega:**
   * Clientes relataram explicitamente lentidão e atraso no recebimento dos itens.
   * *Evidências:* [Review ID: 95e7f613e3dfe80a79306a8c80084739] ("Muita demora na entrega") e [Review ID: 65e3d04aef73660b0b2145ef3d5762cd] ("A entrega atrasou").

2. **Não entrega do pedido/produto:**
   * Há relatos diretos de pedidos que simplesmente não foram entregues aos compradores.
   * *Evidências:* [Review ID: a83253201bb3ee4c020d9b42f1cfac38] ("Não entregou o pedido.") e [Review ID: e8d1bc0188ba7974e847024cab472cdb] ("Não foi entregue o produto .").

3. **Falta de informação, ausência de prazos e rastreamento vago:**
   * Além do não recebimento da mercadoria, foi apontada a ausência de suporte informativo sobre o status do pedido, com o rastreamento indicando apenas "dificuldades na entrega" sem estipula

In [ ]:
pergunta = "Quais aspectos da experiência de compra são mais elogiados pelos clientes?"
resposta = rag_chain.invoke(pergunta)
print(resposta)

c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Com base nas avaliações fornecidas, os clientes elogiam os seguintes aspectos da experiência de compra:

* **Qualidade geral da loja e do atendimento:** Os clientes destacam a excelência da loja em geral [Review ID: 029f6693f6fa41f1acd19f729e4e8b39], fazem elogios e agradecimentos à experiência [Review ID: bf17681b8e58780d5064140aafa8f74d], e mencionam especificamente que o atendimento é ótimo [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
* **Agilidade na entrega:** A rapidez na entrega é apontada como um ponto positivo da experiência [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
* **Confiança e recomendação:** Há relatos de satisfação que geram maior confiança nas compras online [Review ID: e141ad15361f70095fda499096a56ffc], além de clientes que afirmam já ter recomendado a loja para amigos e familiares [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16] ou que indicam o site com frequência [Review ID: d6611f3952b75988e786017dcb297995].


In [ ]:
pergunta = "Quais são os principais motivos de insatisfação relatados nas avaliações?"
resposta = rag_chain.invoke(pergunta)
print(resposta)

c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Com base nas avaliações fornecidas no contexto, os principais motivos de insatisfação relatados pelos clientes são:

* **Insatisfação geral com o produto e expectativas não atendidas:** Clientes relatam insatisfação com o produto adquirido, afirmando que ele não corresponde às expectativas [Review ID: d95962b1b4c23be31196b750630d00d6; Review ID: 87fb462b15995bf3c5b6fba9d9678e04], que não cumpriu com o prometido [Review ID: b2fa7acac4cbfa62f9991421ff55545f] e que apresenta "péssima qualidade" [Review ID: 8d0f11dad4a5706b1a970995c6585605].
* **Problemas de comunicação e rastreio:** Há relatos sobre a falta de atualização do status do pedido e falha na comunicação da empresa com o cliente [Review ID: cb30c1d49c50132378faa2a855dbbb6d].
* **Atraso ou não recebimento do produto:** Cliente relata que já realizou o pagamento (da primeira parcela), mas até o momento não recebeu o produto comprado [Review ID: 8d0f11dad4a5706b1a970995c6585605].


Teste de Guardrails 

In [ ]:
pergunta_fora = "Qual a previsão do tempo para amanhã em São Paulo?"
resposta_fora = rag_chain.invoke(pergunta_fora)
print(resposta_fora)

c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Não foram encontradas avaliações suficientes na base de conhecimento sobre este assunto.


## Desafio Adicional: Metadados e Exploração Contextual

In [36]:
def consultar_voz_do_cliente(
    pergunta: str,
    nota_maxima: int = None,
    nota_minima: int = None,
    apenas_com_atraso: bool = None,
    categoria: str = None,
    k: int = 5
):
    """
    Realiza uma consulta no RAG aplicando filtros contextuais de metadados no ChromaDB.
    """
    condicoes = []
    
    # Filtro por nota máxima (ex: <= 2 para clientes insatisfeitos)
    if nota_maxima is not None:
        condicoes.append({"review_score": {"$lte": int(nota_maxima)}})
        
    # Filtro por nota mínima (ex: >= 4 para elogios)
    if nota_minima is not None:
        condicoes.append({"review_score": {"$gte": int(nota_minima)}})
        
    # Filtro logístico de entrega com atraso
    if apenas_com_atraso is not None:
        condicoes.append({"entregue_com_atraso": {"$eq": bool(apenas_com_atraso)}})
        
    # Filtro por categoria de produto
    if categoria is not None:
        condicoes.append({"product_category_name": {"$eq": str(categoria)}})
    
    # Construir o filtro na sintaxe do ChromaDB
    filtro_chroma = None
    if len(condicoes) == 1:
        filtro_chroma = condicoes[0]
    elif len(condicoes) > 1:
        filtro_chroma = {"$and": condicoes}
    
    # Configurar o retriever dinâmico
    search_kwargs = {"k": k}
    if filtro_chroma:
        search_kwargs["filter"] = filtro_chroma
        
    retriever_dinamico = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs
    )
    
    # Montar a chain com o retriever filtrado
    chain_dinamica = (
        {"context": retriever_dinamico | formatar_documentos, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    print(f" Executando busca com filtros: {filtro_chroma}")
    return chain_dinamica.invoke(pergunta)


In [ ]:
resposta_insatisfacao = consultar_voz_do_cliente(
    pergunta="Quais são as principais queixas sobre a qualidade ou funcionamento dos produtos?",
    nota_maxima=2,   
    k=5
)
print("--- INSATISFAÇÃO (NOTAS BAIXAS) ---")
print(resposta_insatisfacao)

 Executando busca com filtros: {'review_score': {'$lte': 2}}


c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- INSATISFAÇÃO (NOTAS BAIXAS) ---
Com base nas avaliações fornecidas, as principais queixas dos clientes sobre a qualidade ou funcionamento dos produtos concentram-se em insatisfação geral com as expectativas, baixa qualidade e problemas de acabamento. Abaixo estão as evidências detalhadas:

* **Insatisfação geral com o produto:** Clientes relatam que o item não atende às expectativas ou não corresponde ao que é mostrado nas imagens.
  * *Evidências:* [Review ID: 87fb462b15995bf3c5b6fba9d9678e04], [Review ID: 3b3d81110e937a8724dd5e93e020b00d]

* **Baixa qualidade e problemas de acabamento:** Há reclamações explícitas sobre a má qualidade dos materiais e acabamentos ruins, sendo os produtos descritos como finos e frágeis.
  * *Evidências:* [Review ID: 074c39cb5e90578eaeb560e6f632ae1e], [Review ID: 3b3d81110e937a8724dd5e93e020b00d], [Review ID: 8d0f11dad4a5706b1a970995c6585605]


In [ ]:
resposta_elogios = consultar_voz_do_cliente(
    pergunta="O que os clientes mais valorizam e elogiam na experiência de compra?",
    nota_minima=5,  # Apenas notas 5 estrelas
    k=5
)
print("--- ELOGIOS (NOTA MÁXIMA) ---")
print(resposta_elogios)

 Executando busca com filtros: {'review_score': {'$gte': 5}}


c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- ELOGIOS (NOTA MÁXIMA) ---
Com base nas avaliações fornecidas, os clientes valorizam e elogiam os seguintes aspectos da experiência de compra na Olist:

1. **Qualidade geral da loja e satisfação com a compra:** Os clientes demonstram grande satisfação geral, expressando parabéns, elogios e afirmando que gostaram muito de comprar na loja [Review ID: 029f6693f6fa41f1acd19f729e4e8b39, Review ID: bf17681b8e58780d5064140aafa8f74d, Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
2. **Bom atendimento e rapidez na entrega:** Há destaque específico para a qualidade do atendimento e a agilidade na entrega dos produtos [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
3. **Recomendação espontânea a amigos, familiares e conhecidos:** A confiança e a satisfação são tão altas que os clientes indicam a loja para outras pessoas e realizam compras recorrentes no site [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16, Review ID: d6611f3952b75988e786017dcb297995].
4. **Confiabilidade no e-commerce:** A experiên

In [ ]:
resposta_logistica = consultar_voz_do_cliente(
    pergunta="Quais os principais transtornos causados pelo atraso na entrega?",
    apenas_com_atraso=True,  # Apenas pedidos com atraso confirmado
    k=5
)
print("--- LOGÍSTICA & ATRASOS ---")
print(resposta_logistica)

 Executando busca com filtros: {'entregue_com_atraso': {'$eq': True}}


c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- LOGÍSTICA & ATRASOS ---
Com base nas avaliações fornecidas, os principais transtornos causados pelo atraso na entrega incluem:

* **Falta de recebimento do produto e prazos excessivos:** Clientes relatam que os produtos não foram entregues, mesmo com prazos já considerados absurdos [Review ID: b7fe825a931a313b8aa2d18e8004b328], além de muita demora geral no serviço [Review ID: 95e7f613e3dfe80a79306a8c80084739].
* **Prejuízo financeiro e impacto profissional:** Uma cliente relata estar tendo prejuízo financeiro pois o atraso impede que sua esposa (nutricionista) atenda sua clientela, já que ela depende do produto para trabalhar [Review ID: cf6506ccf9e5680269e733ff320d514d].
* **Falta de informação e transparência (descaso):** Os clientes queixam-se da ausência de informações sobre o paradeiro da encomenda [Review ID: cbc6f7cd85fd98b26a724ccb2e9c74b2]. Quando há algum status na rastreabilidade (como "dificuldades na entrega"), ele é vago e não fornece um novo prazo [Review ID: cbc6f7